![Teste vocacional com Machine Learning](capa.png)

# Projeto - Teste Vocacional com Machine Learning
## Etapa 04 - Modelagem

Nesta etapa, os escores RIASEC são usados para treinar modelos de classificação que **recomendam as 3 áreas de formação mais compatíveis** com o perfil de cada pessoa.

**Entrada:** `dados/Dados processados/dados_limpos.csv` (gerado pelo `02_limpeza.ipynb`)

**O que a EDA (etapa 03) ensinou para esta etapa:**
- A base é desbalanceada. Linha de base: **28,62%** no top-1 e **62,61%** no top-3.
- As áreas se sobrepõem bastante: o modelo precisa usar as 6 dimensões juntas, e o top-1 não vai ser muito alto.
- Nenhuma correlação entre as dimensões passa de 0,8: as 6 entram no modelo.
- A coluna `genero` **não** entra no modelo, por causa do risco de viés.

**Métricas:**
- **Acurácia top-3** (principal): a área real da pessoa está entre as 3 recomendadas?
- **F1 macro**: dá o mesmo peso a todas as áreas, para não premiar um modelo que só acerta as áreas grandes.
- **Acurácia top-1**: como referência.

**Perguntas que esta etapa responde:**
1. Algum modelo supera a linha de base com folga?
2. Qual modelo tem o melhor desempenho no top-3 e no F1 macro?
3. Ajustar os hiperparâmetros melhora o modelo escolhido?

In [9]:
# Versão da Linguagem Python
from platform import python_version
print('Versão da Linguagem Python usada neste Projeto:', python_version())

Versão da Linguagem Python usada neste Projeto: 3.14.4


In [10]:
# Os pacotes deste projeto estão listados no requirements.txt, na raiz do projeto.
# Para instalar, no terminal do VS Code, com o ambiente virtual ativado:
#
#   .venv\Scripts\Activate.ps1
#   pip install -r requirements.txt
#
# Pacotes novos desta etapa (instalar uma vez e depois atualizar o requirements.txt):
#
#   pip install scikit-learn xgboost
#   pip freeze > requirements.txt
#
# Depois de instalar ou atualizar um pacote, reinicie o kernel do notebook.

### Importando as bibliotecas necessárias

In [11]:
# Para manipulação dos dados - a dupla inseparável
import pandas as pd
import numpy as np

# Para visualização dos dados
import matplotlib.pyplot as plt
import seaborn as sns

# As ferramentas do scikit-learn e do XGBoost serão importadas aqui, à medida que forem usadas

In [12]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "Gustavo" --iversions

Author: Gustavo

matplotlib: 3.11.2
numpy     : 2.5.3
pandas    : 3.0.6
platform  : 1.0.9
seaborn   : 0.13.2
sklearn   : 1.9.1



### Importação do conjunto de dados tratado

In [16]:
# Importando o dataset tratado na etapa de limpeza
df = pd.read_csv('../dados/Dados processados/dados_limpos.csv')

# Formato do dataset: devem ser 81.692 participantes e 101 colunas
df.shape

(81692, 101)

In [14]:
# As 6 dimensões do modelo de Holland e as colunas de escore correspondentes
dimensoes = {
    'R': 'Realista',
    'I': 'Investigativo',
    'A': 'Artístico',
    'S': 'Social',
    'E': 'Empreendedor',
    'C': 'Convencional',
}
colunas_escores = [f'escore_{letra}' for letra in dimensoes]

---
## 1. Preparação dos dados

**Objetivo:** separar o que o modelo vai **usar para prever** (as variáveis explicativas, `X`) do que ele vai **prever** (o alvo, `y`).

- `X`: os 6 escores RIASEC (`colunas_escores`). São as mesmas informações que o formulário do site vai coletar.
- `y`: a coluna `area`.

Antes disso, é preciso decidir o que fazer com as áreas pequenas: **Serviços (192)** e **Agricultura e veterinária (141)**.


decidir o que fazer com as áreas pequenas
Rodei uma regressão logística rápida aqui, com e sem as áreas pequenas, para você decidir com números:

Top-1	Top-3	F1 macro
Com todas as 10 áreas	37,95%	73,32%	0,181
Sem Serviços e Agricultura	38,05%	73,61%	0,226
O top-1 e o top-3 quase não mudam, porque elas são só 0,4% da base. Mas o F1 macro sobe bastante. O F1 macro dá o mesmo peso a todas as áreas, e o modelo nunca acerta essas duas (são 141 e 192 exemplos contra 23 mil de Ciências sociais), então elas puxam a média para baixo.

In [18]:
# Áreas com poucos participantes para o modelo aprender (141 e 192 pessoas, 0,4% da base)
areas_pequenas = ['Serviços', 'Agricultura e veterinária']

# O ~ inverte o filtro: mantém quem NÃO está nas áreas pequenas
df_modelo = df[~df['area'].isin(areas_pequenas)]

# Conferindo: devem sobrar 8 áreas e 81.359 participantes
print(df_modelo['area'].nunique(), 'áreas')
print(df_modelo.shape)


8 áreas
(81359, 101)


In [19]:
# X: o que o modelo usa para prever (os 6 escores, as mesmas respostas que o formulário do site vai coletar)
X = df_modelo[colunas_escores]

# y: o que o modelo vai prever (a área de formação)
y = df_modelo['area']

# Gênero guardado à parte: NÃO entra no modelo, só vai ser usado na avaliação por gênero (etapa 05)
genero = df_modelo['genero']

X.shape, y.shape


((81359, 6), (81359,))

---
## 2. Divisão em treino e teste

**Objetivo:** separar uma parte dos dados que o modelo **nunca vê** durante o treino. É com ela que medimos se o modelo aprendeu de verdade ou só decorou.

> ✏️ **Sua vez:**
> - Use o `train_test_split` do `sklearn.model_selection`, com 20% para teste (`test_size=0.2`).
> - Use `random_state=42`, para a divisão ser sempre a mesma (e a etapa 05 conseguir recriá-la).
> - Use `stratify=y`. Ele garante que a proporção de cada área seja a mesma no treino e no teste.
> - Confira o tamanho de cada parte com `.shape`, e a proporção das áreas com `value_counts(normalize=True)`.
>
> 🤔 **Para pensar:** o que poderia acontecer com Agricultura (141 pessoas) se a divisão fosse feita sem o `stratify`?

In [20]:
from sklearn.model_selection import train_test_split

# Divide X e y em treino (80%) e teste (20%), mantendo as linhas casadas
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=0.2,       # 20% para o teste
    random_state=42,     # semente fixa: a divisão é sempre a mesma toda vez que rodar
    stratify=y           # mantém a mesma proporção de cada área no treino e no teste
)

print('Treino:', X_treino.shape)
print('Teste: ', X_teste.shape)


Treino: (65087, 6)
Teste:  (16272, 6)


In [21]:
# Conferindo o stratify: as porcentagens de cada área devem ser praticamente iguais
proporcao = pd.concat([
    y_treino.value_counts(normalize=True),
    y_teste.value_counts(normalize=True)
], axis=1).round(4) * 100
proporcao.columns = ['Treino (%)', 'Teste (%)']
proporcao


,Treino (%),Teste (%)
area,,
"Ciências sociais, comunicação e informação",28.73,28.74
"Negócios, administração e direito",20.55,20.55
Artes e humanidades,13.58,13.58
Saúde e bem-estar,11.08,11.08
"Ciências naturais, matemática e estatística",9.32,9.32
"Engenharia, produção e construção",8.73,8.73
Computação e TIC,4.09,4.08
Educação,3.93,3.93


---
## 3. Como avaliar os modelos

**Objetivo:** criar uma função que calcula as métricas de qualquer modelo, para não repetir o mesmo código em cada seção.

> ✏️ **Sua vez:**
> - Crie uma função `avaliar_modelo(nome, modelo, X_teste, y_teste)` que devolve um dicionário com o nome do modelo, a **acurácia top-1**, a **acurácia top-3** e o **F1 macro**.
> - Para o top-3, use o `predict_proba` do modelo (a probabilidade de cada área) com o `top_k_accuracy_score` do `sklearn.metrics` e `k=3`. Passe também `labels=modelo.classes_`.
> - Para o top-1 e o F1, use o `predict` com `accuracy_score` e `f1_score(..., average='macro')`.
>
> 🤔 **Para pensar:** por que a função usa `return` em vez de `print`? (Lembre do desafio da etapa 01.)

---
## 4. Linha de base com o DummyClassifier

**Objetivo:** confirmar a linha de base calculada na EDA, agora com o mesmo processo que os modelos de verdade vão seguir.

O `DummyClassifier` é um modelo "de mentira": com `strategy='most_frequent'`, ele sempre responde a maior área. Qualquer modelo que não o supere não aprendeu nada.

> ✏️ **Sua vez:**
> - Importe o `DummyClassifier` do `sklearn.dummy`.
> - Treine com `.fit(X_treino, y_treino)` e avalie com a sua função.
> - Guarde o resultado numa lista `resultados`, que vai receber o resultado de todos os modelos.
>
> 🤔 **Para pensar:** os valores bateram com os da EDA (28,62% e 62,61%)? Se ficaram um pouco diferentes, por quê?

---
## 5. Regressão logística

**Objetivo:** treinar o primeiro modelo de verdade. A regressão logística é simples, rápida e fácil de explicar. Por isso é o ponto de partida.

> ✏️ **Sua vez:**
> - Importe a `LogisticRegression` do `sklearn.linear_model`.
> - Use `max_iter=1000` (o padrão às vezes não é suficiente para o modelo terminar de aprender).
> - Treine, avalie e adicione o resultado à lista `resultados`.
>
> 🤔 **Para pensar:** apesar do nome, a regressão logística é um modelo de **classificação**. Por que você acha que ela tem "regressão" no nome?

---
## 6. KNN (k vizinhos mais próximos)

**Objetivo:** testar um modelo que funciona por **semelhança**: para uma pessoa nova, ele procura as `k` pessoas com escores mais parecidos e vê quais áreas elas cursam.

> ✏️ **Sua vez:**
> - Importe o `KNeighborsClassifier` do `sklearn.neighbors`.
> - Comece com `n_neighbors=50`.
> - Treine, avalie e adicione o resultado à lista `resultados`.
>
> 🤔 **Para pensar:** o KNN mede a distância entre as pessoas. Em geral, é preciso padronizar as variáveis antes de usá-lo. Por que aqui isso não é tão necessário? (Dica: em que escala estão os 6 escores?)

---
## 7. Random Forest (floresta aleatória)

**Objetivo:** testar um modelo baseado em **árvores de decisão**. A floresta treina muitas árvores diferentes e junta as respostas delas.

> ✏️ **Sua vez:**
> - Importe o `RandomForestClassifier` do `sklearn.ensemble`.
> - Use `n_estimators=200`, `min_samples_leaf=20`, `random_state=42` e `n_jobs=-1` (usa todos os núcleos do processador).
> - Treine, avalie e adicione o resultado à lista `resultados`.
>
> 🤔 **Para pensar:** o `min_samples_leaf=20` impede que uma árvore crie uma regra para menos de 20 pessoas. Por que isso ajuda a evitar que o modelo "decore" o treino?

---
## 8. XGBoost

**Objetivo:** testar um modelo de **boosting**, em que cada árvore nova tenta corrigir os erros das anteriores. Costuma ser um dos modelos mais fortes em dados de tabela.

O XGBoost só aceita o alvo em **números** (0, 1, 2...), e não em texto. Por isso, é preciso converter as áreas antes.

> ✏️ **Sua vez:**
> - Converta o `y_treino` e o `y_teste` em números com o `LabelEncoder` do `sklearn.preprocessing` (`fit_transform` no treino, `transform` no teste).
> - Importe o `XGBClassifier` do `xgboost`, com `n_estimators=300`, `learning_rate=0.1`, `max_depth=4` e `random_state=42`.
> - Treine, avalie e adicione o resultado à lista `resultados`.
>
> 🤔 **Para pensar:** por que o `LabelEncoder` usa `fit_transform` no treino e só `transform` no teste?

---
## 9. Comparação dos modelos

**Objetivo:** colocar todos os resultados lado a lado e escolher o melhor modelo.

> ✏️ **Sua vez:**
> - Transforme a lista `resultados` num DataFrame e ordene pela acurácia top-3.
> - Faça um gráfico de barras comparando o top-3 dos modelos, com uma linha marcando a linha de base.
>
> 🤔 **Para pensar:** se dois modelos tiverem resultados muito próximos, qual você escolheria: o mais simples ou o mais complexo? Pense também no deploy (tamanho do arquivo e facilidade de explicar o resultado).

---
## 10. Ajuste de hiperparâmetros do modelo escolhido

**Objetivo:** testar combinações de hiperparâmetros do melhor modelo para ver se ele melhora.

Os hiperparâmetros são as "configurações" do modelo que nós escolhemos (como o `n_neighbors` do KNN ou o `max_depth` do XGBoost). Ele não aprende esses valores sozinho.

> ✏️ **Sua vez:**
> - Use o `GridSearchCV` do `sklearn.model_selection` com algumas combinações de hiperparâmetros do modelo escolhido.
> - Use `cv=5` (validação cruzada em 5 partes) e `scoring='f1_macro'`.
> - Treine **só com os dados de treino**. Depois, avalie o melhor modelo (`.best_estimator_`) no teste.
>
> 🤔 **Para pensar:** por que o ajuste usa só o treino? O que aconteceria se usássemos o teste para escolher os hiperparâmetros?

---
## 11. Salvando o modelo final

**Objetivo:** salvar o modelo escolhido num arquivo, para ser usado na avaliação (etapa 05) e no site com Flask, igual ao `model.joblib` do curso.

> ✏️ **Sua vez:**
> - Use o `joblib.dump` para salvar o modelo final em `../modelos/modelo_final.joblib`.
> - Crie antes a pasta `modelos/` na raiz do projeto.
>
> 🤔 **Para pensar:** o arquivo do modelo deve ir para o GitHub? Pense no tamanho do arquivo e no fato de que ele pode ser recriado rodando este notebook.

---
## Conclusões da modelagem

> ✏️ Escreva aqui, com as suas palavras, o que esta etapa mostrou. Algumas perguntas para guiar:
> - Os modelos superaram a linha de base? Por quanto?
> - Qual foi o melhor modelo, e por que ele foi escolhido?
> - O ajuste de hiperparâmetros ajudou?
> - O que ficou para a etapa de avaliação (matriz de confusão, avaliação por gênero, SHAP)?

## Fim da Etapa 04